In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("Gautam_buddha.txt")
documents = loader.load()
text_splitter =  CharacterTextSplitter(chunk_size = 200, chunk_overlap=30)
docs = text_splitter.split_documents(documents)




In [ ]:
#embeddings = OllamaEmbeddings()  # This didn't work, so had to run on terminal>> \s05_langchain >>  ollama pull nomic-embed-text; and then below line worked.
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db=FAISS.from_documents(docs, embeddings) # now this is our vector store db
db

In [ ]:
### Querying
query = "Who was Siddhartha Gautama?"
docs = db.similarity_search(query)
docs


In [ ]:
# Using Retriever to convert the vestorstore into Retriever Class allowing us to use it on 
# other LangChain methods that work with Retrievers.
# 

retriever = db.as_retriever()
retriever.invoke(query)

In [ ]:
# Similarity search with score: similarity_search_with_score() is method in Faiss that returns document with distance 
# score. The returned score is L2 distance (the lower score the better)
docs_with_sim_score = db.similarity_search_with_score(query)

docs_with_sim_score

In [ ]:
# We can also retrive using directly vectors instead of passing sentences

embedding_vector = embeddings.embed_query (query)
embedding_vector

docs_with_sim_score = db.similarity_search_by_vector(embedding_vector)
docs_with_sim_score


In [ ]:
# Saving vectorstoreDB in local and loading back from local
db.save_local("faiss_index") # this will create folder /faiss_index and files in it names index.faiss and index.pkl

# Loading from local
new_db = FAISS.load_local ("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = new_db.similarity_search(query)
docs